# 基于标准化设备 Log 的 AI 良率异常洞察 Demo

本 Notebook 展示一个从设备 Log 到工程洞察的轻量级 AI 良率分析流程。

核心流程包括：

1. 读取原始设备 Log；
2. 将原始 Log 标准化为统一分析表；
3. 基于标准化数据进行特征选择与清洗；
4. 构造工艺状态特征；
5. 自动识别异常工况；
6. 对比正常段与异常段，定位关键异常关联因子；
7. 使用 LLM 生成工程师可读的异常分析报告。

重要原则：

- 原始数据只用于生成标准化数据；
- 后续所有分析均基于 `data/processed_equipment_log_standardized.xlsx`；
- Python / 统计 / 机器学习负责计算；
- LLM 只负责解释、总结和生成文本报告。

In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Plot settings: avoid minus rendering issues
plt.rcParams["axes.unicode_minus"] = False

# Suppress noisy runtime warnings in demo environments
warnings.filterwarnings(
    "ignore",
    message="Could not infer format, so each element will be parsed individually.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r"Glyph .* missing from font\(s\).*",
    category=UserWarning,
)

CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "data").exists() and (CURRENT_DIR / "notebooks").exists():
    PROJECT_ROOT = CURRENT_DIR
elif CURRENT_DIR.name == "notebooks" and (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    ROOT_CANDIDATES = [CURRENT_DIR, *CURRENT_DIR.parents]
    PROJECT_ROOT = next(
        (p for p in ROOT_CANDIDATES if (p / "data").exists() and (p / "notebooks").exists()),
        CURRENT_DIR,
    )

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

RAW_DATA_PATH = DATA_DIR / "equipment_log.xlsx"
PROCESSED_DATA_PATH = DATA_DIR / "processed_equipment_log_standardized.xlsx"


def safe_to_datetime(series):
    """Parse datetime without noisy infer-format warnings."""
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed")
    except TypeError:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message="Could not infer format, so each element will be parsed individually.*",
                category=UserWarning,
            )
            return pd.to_datetime(series, errors="coerce")

print("当前阶段：1. 环境准备与路径配置")
print("当前工作目录：", CURRENT_DIR)
print("项目根目录：", PROJECT_ROOT)
print("数据目录：", DATA_DIR.resolve())
print("原始数据文件：", RAW_DATA_PATH)
print("标准化数据文件：", PROCESSED_DATA_PATH)

In [ ]:
print("当前阶段：2. 原始设备 Log 数据读取与结构检查")
print(f"输入数据：{RAW_DATA_PATH}")

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"未找到原始设备 Log 文件：{RAW_DATA_PATH}\n"
        "请确认文件位于与 notebooks 同级的 data 目录（例如 ../data/equipment_log.xlsx）。"
    )

df_raw = pd.read_excel(RAW_DATA_PATH)

print(f"原始数据维度：{df_raw.shape[0]} 行 × {df_raw.shape[1]} 列")
print("原始字段示例：", df_raw.columns[:12].tolist())

duplicate_cols = pd.Series(df_raw.columns).value_counts()
duplicate_cols = duplicate_cols[duplicate_cols > 1]
print(f"重复字段数量：{duplicate_cols.shape[0]}")
if not duplicate_cols.empty:
    display(duplicate_cols.to_frame("count").head(20))

l1_cols = [c for c in df_raw.columns if str(c).startswith("L1")]
l2_cols = [c for c in df_raw.columns if str(c).startswith("L2")]
print(f"L1 字段数量：{len(l1_cols)}")
print(f"L2 字段数量：{len(l2_cols)}")
if l1_cols and l2_cols:
    l1_suffix = {c[2:] for c in l1_cols}
    l2_suffix = {c[2:] for c in l2_cols}
    print(f"L2 相对 L1 额外字段数量：{len(l2_suffix - l1_suffix)}")
    print(f"L1 相对 L2 额外字段数量：{len(l1_suffix - l2_suffix)}")

display(df_raw.head())

In [ ]:
print("当前阶段：3. 数据标准化处理层")
print(f"输入数据：{RAW_DATA_PATH}")
print(f"输出数据：{PROCESSED_DATA_PATH}")

def make_unique_columns(columns):
    seen = {}
    out = []
    for col in columns:
        col = str(col).strip()
        if col not in seen:
            seen[col] = 0
            out.append(col)
        else:
            seen[col] += 1
            out.append(f"{col}__dup{seen[col]}")
    return out

def normalize_name(name):
    s = str(name).strip()
    s = s.replace("（", "(").replace("）", ")")
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^0-9a-zA-Z_\u4e00-\u9fa5]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def infer_time_columns(df):
    keywords = ["time", "date", "timestamp", "datetime", "record", "log"]
    return [c for c in df.columns if any(k in str(c).lower() for k in keywords)]

def coerce_numeric_columns(df, ratio_threshold=0.8):
    converted = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
        s = df[col]
        if not (pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s)):
            continue
        cleaned = s.astype(str).str.replace(",", "", regex=False).str.strip()
        numeric = pd.to_numeric(cleaned, errors="coerce")
        if numeric.notna().mean() >= ratio_threshold:
            df[col] = numeric
            converted.append(col)
    return converted

df_standardized = df_raw.copy()
df_standardized.columns = make_unique_columns(df_standardized.columns)
df_standardized.columns = [normalize_name(c) for c in df_standardized.columns]
df_standardized.columns = make_unique_columns(df_standardized.columns)

all_empty_cols = [c for c in df_standardized.columns if df_standardized[c].isna().all()]
if all_empty_cols:
    df_standardized = df_standardized.drop(columns=all_empty_cols)

candidate_time_cols = infer_time_columns(df_standardized)
parsed_time_cols = []
for c in candidate_time_cols:
    parsed = safe_to_datetime(df_standardized[c])
    if parsed.notna().mean() >= 0.5:
        df_standardized[c] = parsed
        parsed_time_cols.append(c)

converted_numeric_cols = coerce_numeric_columns(df_standardized, ratio_threshold=0.8)

rename_map = {}
for c in df_standardized.columns:
    rename_map[c] = re.sub(r"^l(\d+)_", lambda m: f"L{m.group(1)}_", c, flags=re.IGNORECASE)
df_standardized = df_standardized.rename(columns=rename_map)
df_standardized.columns = make_unique_columns(df_standardized.columns)

df_standardized.to_excel(PROCESSED_DATA_PATH, index=False)

print(f"标准化数据已保存：{PROCESSED_DATA_PATH}")
print(f"标准化数据维度：{df_standardized.shape[0]} 行 × {df_standardized.shape[1]} 列")
print(f"删除全空列数量：{len(all_empty_cols)}")
print(f"识别并解析时间列数量：{len(parsed_time_cols)}")
print(f"转换为数值列数量：{len(converted_numeric_cols)}")
display(df_standardized.head())

In [ ]:
print("当前阶段：4. 标准化数据读取与质量检查")
print(f"输入数据：{PROCESSED_DATA_PATH}")

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"未找到标准化后的数据文件：{PROCESSED_DATA_PATH}\n"
        "请先运行数据标准化处理层，生成 processed_equipment_log_standardized.xlsx。"
    )

df_standardized = pd.read_excel(PROCESSED_DATA_PATH)

print(f"使用标准化后的数据文件：{PROCESSED_DATA_PATH}")
print(f"标准化数据维度：{df_standardized.shape[0]} 行 × {df_standardized.shape[1]} 列")
print(f"字段数量：{df_standardized.shape[1]}")

missing_summary = df_standardized.isna().mean().sort_values(ascending=False).head(20)

numeric_cols_all = df_standardized.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols_all = [c for c in df_standardized.columns if c not in numeric_cols_all]

dup_check = pd.Series(df_standardized.columns).value_counts()
dup_check = dup_check[dup_check > 1]

time_cols_qc = []
for c in df_standardized.columns:
    if "time" in str(c).lower() or "date" in str(c).lower():
        parsed = safe_to_datetime(df_standardized[c])
        if parsed.notna().mean() > 0.5:
            time_cols_qc.append(c)

print(f"数值列数量：{len(numeric_cols_all)}")
print(f"非数值列数量：{len(non_numeric_cols_all)}")
print(f"重复列数量：{dup_check.shape[0]}")
print(f"可用时间列数量：{len(time_cols_qc)}")

display(df_standardized.head())
display(missing_summary.to_frame("missing_ratio"))

In [ ]:
print("当前阶段：5. 特征选择与清洗层")
print(f"输入数据：{PROCESSED_DATA_PATH}")

df_clean = df_standardized.copy()
df_clean = df_clean.dropna(axis=1, how="all")

possible_time_cols = [c for c in df_clean.columns if any(k in str(c).lower() for k in ["time", "date", "timestamp"])]
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

constant_cols = [col for col in numeric_cols if df_clean[col].nunique(dropna=True) <= 1]
if constant_cols:
    df_clean = df_clean.drop(columns=constant_cols)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
missing_ratio = df_clean[numeric_cols].isna().mean() if numeric_cols else pd.Series(dtype=float)
feature_cols = missing_ratio[missing_ratio < 0.5].index.tolist() if not missing_ratio.empty else []

if not feature_cols:
    raise ValueError("清洗后没有可用数值特征，请检查原始数据或放宽筛选阈值。")

X_base = df_clean[feature_cols].copy()
X_base = X_base.replace([np.inf, -np.inf], np.nan)
X_base = X_base.fillna(X_base.median(numeric_only=True))

print(f"清洗后数据维度：{df_clean.shape}")
print(f"候选数值特征数量：{len(feature_cols)}")
print(f"删除常数列数量：{len(constant_cols)}")
print(f"识别的时间列候选数量：{len(possible_time_cols)}")
print("前 20 个候选特征：", feature_cols[:20])

In [ ]:
print("当前阶段：6. 特征工程层")
print(f"输入特征数量：{len(feature_cols)}")

from sklearn.preprocessing import StandardScaler

feature_df = df_clean.copy()
X_base = X_base.replace([np.inf, -np.inf], np.nan)
X_base = X_base.fillna(X_base.median(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_base)
X = pd.DataFrame(X_scaled, columns=feature_cols, index=X_base.index)

TOP_N_FOR_ROLLING = min(20, len(feature_cols))
rolling_base_cols = feature_cols[:TOP_N_FOR_ROLLING]
for col in rolling_base_cols:
    X[f"{col}__diff1"] = X[col].diff().fillna(0.0)
    X[f"{col}__roll_mean_5"] = X[col].rolling(window=5, min_periods=1).mean()
    X[f"{col}__roll_std_5"] = X[col].rolling(window=5, min_periods=1).std().fillna(0.0)

engineered_feature_cols = X.columns.tolist()

print(f"特征工程后特征矩阵维度：{X.shape}")
print(f"工程化特征总数：{len(engineered_feature_cols)}")

In [ ]:
print("当前阶段：7. 异常检测层")
print("输入数据：特征工程输出矩阵 X")

from sklearn.ensemble import IsolationForest

model = IsolationForest(n_estimators=200, contamination=0.1, random_state=42)
anomaly_pred = model.fit_predict(X)
anomaly_score = model.decision_function(X)

df_anomaly = feature_df.copy()
df_anomaly["anomaly_label"] = np.where(anomaly_pred == -1, 1, 0)
df_anomaly["anomaly_score"] = anomaly_score

print("异常检测完成")
print(df_anomaly["anomaly_label"].value_counts())
print(f"异常比例：{df_anomaly['anomaly_label'].mean():.2%}")

In [ ]:
print("当前阶段：8. 分段对比与核心洞察层")
print("输入数据：df_anomaly + engineered_feature_cols")

normal_df = df_anomaly[df_anomaly["anomaly_label"] == 0]
abnormal_df = df_anomaly[df_anomaly["anomaly_label"] == 1]

comparison_rows = []
base_compare_cols = [c for c in feature_cols if c in df_anomaly.columns]

for col in base_compare_cols:
    normal_mean = normal_df[col].mean()
    abnormal_mean = abnormal_df[col].mean()
    normal_std = normal_df[col].std()
    abnormal_std = abnormal_df[col].std()
    diff = abnormal_mean - normal_mean
    pct_change = diff / (abs(normal_mean) + 1e-9)

    comparison_rows.append({
        "feature": col,
        "normal_mean": normal_mean,
        "abnormal_mean": abnormal_mean,
        "mean_diff": diff,
        "pct_change": pct_change,
        "normal_std": normal_std,
        "abnormal_std": abnormal_std,
        "std_diff": abnormal_std - normal_std,
        "abs_pct_change": abs(pct_change)
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("abs_pct_change", ascending=False)
top_factors = comparison_df.head(10).copy()
insight_table = top_factors.copy()

print(f"参与对比特征数量：{len(base_compare_cols)}")
print("Top 10 异常关联因子：")
display(top_factors)

In [ ]:
print("当前阶段：9. 可视化展示层")
print("输入数据：df_anomaly / comparison_df / top_factors")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].hist(df_anomaly["anomaly_score"], bins=30, color="#4C78A8", alpha=0.85)
axes[0, 0].set_title("Anomaly score distribution")
axes[0, 0].set_xlabel("anomaly_score")
axes[0, 0].set_ylabel("count")

label_counts = df_anomaly["anomaly_label"].value_counts().sort_index()
axes[0, 1].bar(["Normal(0)", "Abnormal(1)"], [label_counts.get(0, 0), label_counts.get(1, 0)], color=["#72B7B2", "#E45756"])
axes[0, 1].set_title("Normal vs abnormal sample count")

plot_df = top_factors.sort_values("abs_pct_change", ascending=True)
axes[1, 0].barh(plot_df["feature"], plot_df["abs_pct_change"], color="#F58518")
axes[1, 0].set_title("Top 10 differential factors (abs pct change)")
axes[1, 0].set_xlabel("abs_pct_change")

time_col = None
for c in df_anomaly.columns:
    if any(k in str(c).lower() for k in ["time", "date", "timestamp"]):
        parsed = safe_to_datetime(df_anomaly[c])
        if parsed.notna().mean() > 0.5:
            time_col = c
            break

if time_col is not None:
    temp = df_anomaly.copy()
    temp[time_col] = safe_to_datetime(temp[time_col])
    temp = temp.dropna(subset=[time_col]).sort_values(time_col)
    axes[1, 1].plot(temp[time_col], temp["anomaly_label"], marker=".", linestyle="", alpha=0.5)
    axes[1, 1].set_title(f"Anomaly timeline ({time_col})")
    axes[1, 1].set_xlabel("time")
    axes[1, 1].set_ylabel("anomaly_label")
else:
    axes[1, 1].text(0.1, 0.5, "No usable time column detected", fontsize=12)
    axes[1, 1].set_title("Anomaly over time")
    axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("当前阶段：10. AI 文本洞察生成层")
print("输入数据：analysis_summary（结构化结果）")

analysis_summary = {
    "data_shape": df_standardized.shape,
    "clean_shape": df_clean.shape,
    "feature_count": len(feature_cols),
    "anomaly_count": int(df_anomaly["anomaly_label"].sum()),
    "anomaly_ratio": float(df_anomaly["anomaly_label"].mean()),
    "top_factors": top_factors.to_dict(orient="records"),
    "missing_top10": df_standardized.isna().mean().sort_values(ascending=False).head(10).round(4).to_dict()
}

prompt = (
    "请基于以下结构化分析结果生成‘AI 良率异常洞察报告’。\n"
    "要求：\n"
    "1) 不要编造未计算出的数据；\n"
    "2) 不要直接给出因果结论；\n"
    "3) 使用‘可能相关/优先排查/建议进一步验证’等措辞；\n"
    "4) 报告受众为工程师和管理者；\n"
    "5) 使用如下结构：异常概况、关键异常关联因子、工艺解释、优先排查建议、风险提示、下一步数据补充建议。\n\n"
    f"结构化输入：\n{analysis_summary}"
)

ai_report = None
try:
    import sys
    sys.path.append(str(Path("src").resolve()))
    from llm_provider import generate_insight

    llm_result = generate_insight(prompt=prompt, mode="auto")
    ai_report = llm_result.get("report_md") or llm_result.get("text")
    if ai_report:
        print("LLM 生成成功（若不可用将自动回退）。")
        print(ai_report)
except Exception as e:
    print(f"LLM 调用不可用，进入规则生成 fallback：{e}")

if not ai_report:
    fallback_report = f"""
## AI 良率异常洞察报告（规则生成版）

本次分析基于标准化后的设备 Log 数据，共包含 {df_standardized.shape[0]} 行、{df_standardized.shape[1]} 列。
经过特征清洗后，保留 {len(feature_cols)} 个候选工艺特征。

异常检测识别出 {int(df_anomaly["anomaly_label"].sum())} 条疑似异常记录，
异常比例为 {df_anomaly["anomaly_label"].mean():.2%}。

从正常段与异常段对比看，以下参数差异最明显：

{top_factors[["feature", "normal_mean", "abnormal_mean", "pct_change"]].to_markdown(index=False)}

这些参数可作为后续工程排查的优先方向。
需要注意的是，本分析结果表示异常相关性，不等同于严格因果结论。
"""
    ai_report = fallback_report
    print(ai_report)

## 11) 总结与下一步扩展

本 Notebook 已完成如下闭环：

```text
原始设备 Log
    → 标准化数据表
    → 特征清洗
    → 特征工程
    → 异常检测
    → 分段对比
    → AI 洞察报告
```

下一步可扩展方向：

- 接入真实良率标签；
- 接入批次号 / 产品型号 / 设备号；
- 接入班次 / 操作员 / 药液批次；
- 从异常检测升级为良率预测；
- 从单次分析升级为每日自动报告；
- 与 MES / ERP / 数据平台集成；
- 沉淀为可复用的 AI 良率分析模块。